In [1]:
# Load environment variables in a file called .env

import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [2]:
# Step 1: Create your prompts

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""
user_prompt = """
Here are the contents of a document
    Databricks Certified Data Engineer Professional — Complete Study
Guide (Expanded)
# Databricks Certified Data Engineer Professional — Complete Study Guide (Expanded)
This document is an expanded, exam-focused, hands-on friendly guide covering all essential topics for the Databricks
Certified Data Engineer Professional exam. It includes concepts, code-level examples, layman analogies, exam tips,
and practical checklists.
---
## PART 1 — FOUNDATIONS & ARCHITECTURE
### Lakehouse & Databricks Overview
- **Lakehouse** merges data lake flexibility with data warehouse management. Databricks implements the Lakehouse
using Delta Lake on cloud storage.
- **Control plane** (Databricks-managed): user interface, workspace, job orchestration, Unity Catalog control.
**Compute plane** (customer cloud): clusters, DBFS, cloud storage (S3/ADLS/GCS).
**Layman analogy:** Lakehouse = a modern library: raw manuscripts in basement (lake), curated books on shelves
(gold tables), librarian catalog (Unity Catalog).
**Exam tip:** Know the roles of control plane vs compute plane and where metadata lives.
---
### Delta Lake Core Concepts
- **ACID transactions**: transaction log (`_delta_log`) ensures commits are atomic and durable.
- **Delta log**: JSON + Parquet transaction log; time travel uses versions recorded here.
- **Time travel**: `SELECT * FROM table VERSION AS OF n` or `TIMESTAMP AS OF '...'`.
- **VACUUM**: removes old files; retains data for default retention; be careful with retention windows.
- **Schema enforcement** and **schema evolution**: `mergeSchema`, `overwriteSchema`, `addNewColumns`.
- **Optimistic Concurrency Control (OCC)**: concurrent writers create commits, Delta verifies against latest state and
throws conflicts.
**Commands:**
```
DESCRIBE HISTORY mydb.mytable;
RESTORE TABLE mydb.mytable TO VERSION AS OF 3;
VACUUM mydb.mytable RETAIN 168 HOURS;
ALTER TABLE mydb.mytable SET TBLPROPERTIES ('delta.enableChangeDataFeed' = true);
```
**Layman analogy:** Delta log is a journal of changes — like saved drafts + edits history for a document.
---
## PART 2 — CHANGE DATA: CDC & CDF
### CDC (Change Data Capture)
- Implemented typically via `MERGE INTO` operations using source snapshot or CDC flags.
- Pattern: dedupe source, identify inserts/updates/deletes, `MERGE` into target.
**Typical SQL MERGE pattern:**
```
MERGE INTO target t
USING source s
ON t.key = s.key
WHEN MATCHED AND THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```
**Exam tip:** Understand how to pre-process source to avoid multiple matches (use ROW_NUMBER over partition).
### Change Data Feed (CDF)
- Enable per-table: `ALTER TABLE t SET TBLPROPERTIES (delta.enableChangeDataFeed = true)`.
- Read changes:
"""

from openai import OpenAI
openai = OpenAI()

# Step 2: Make the messages list
messages = [{"role": "system", "content": "Wonderful, you are a helpful assistant"},
{"role": "user", "content": "Here is the summary of the document: " + user_prompt}
] # fill this in

# Step 3: Call OpenAI
response = openai.chat.completions.create(model = "gpt-4.1-nano", messages = messages)

# Step 4: print the result
print(response.choices[0].message.content)

Certainly! Based on the provided summary, here is an overview of the key topics covered in the document:

**1. Foundations & Architecture**
- **Lakehouse Concept:** Combines data lake flexibility with data warehouse management using Delta Lake.
- **Databricks Architecture:** Differentiates between the control plane (user interface, workspace, metadata, and orchestration) and compute plane (clusters, storage).
- **Analogy:** Lakehouse as a library with raw manuscripts and curated books.
- **Key Tips:** Understand the roles of control vs compute plane and where metadata resides.

**2. Delta Lake Core Concepts**
- **ACID Transactions & Transaction Log:** Ensures data integrity with `_delta_log`.
- **Time Travel:** Query previous versions of data via version number or timestamp.
- **Vacuum:** Cleanup of obsolete files, with retention policies.
- **Schema Management:** Enforce schema and evolve schema with commands like `mergeSchema` and `addNewColumns`.
- **Optimistic Concurrency Control:*

In [3]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""


def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + website}
    ]



In [4]:
# Let's try out this utility

from scraper import fetch_website_contents

ka= fetch_website_contents("https://blog.khanacademy.org/")
print(ka)

Khan Academy Blog - Khan Academy Blog

Skip to content
Donate
Khan Academy Blog
Khan Academy is a nonprofit with a mission to provide a free, world-class education to anyone, anywhere.
Explore AI in Your Teaching: A One-Hour PD Experience with Khan Academy
November 12, 2025
This December, join educators around the world for Hour of AI—a global event celebrating curiosity, creativity, and …
Read more
Exploring How to Improve Assessment with AI
October 27, 2025
What Happened When Khan Academy Became Its Own School District—and What It Means for Yours
October 22, 2025
Kristen’s Corner Fall 2025
October 22, 2025
Educators
Explore AI in Your Teaching: A One-Hour PD Experience with Khan Academy
November 12, 2025
This December, join educators around the world for Hour of AI—a global event celebrating curiosity, creativity, and the power of teaching with AI. For …
Read more
Kristen’s Corner Fall 2025
October 22, 2025
By Aviv Weiss, Khan Academy Districts Each quarter, we sit down with Dr. Kris

In [5]:
messages_for("https://blog.khanacademy.org/")

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\nhttps://blog.khanacademy.org/'}]

In [6]:
def summarize(url):
    website = fetch_website_contents(url)  #pyright: ignore[reportUndefinedVariable]
    response = openai.chat.completions.create(model = "gpt-4.1-nano", messages = messages_for(website))
    return response.choices[0].message.content


In [7]:
summarize("https://blog.khanacademy.org/")

'# Khan Academy Blog  \nThe site is basically a mild-mannered cheerleader for all things Khan Academy—promoting their mission to offer free, top-tier education worldwide. It highlights upcoming AI-focused events like the "Hour of AI" where teachers get to geek out over teaching with artificial intelligence. Plus, they love to remind everyone about fun classroom tools like LearnStorm and share the latest insights from their big brains (literally, Chief Learning Officer Kristen DiCerbo).  \n\n## News & Announcements  \n- December\'s "Hour of AI" event—because what’s better than celebrating curiosity and creativity with robots?  \n- Deep dive into how Khan Academy became its own school district—sure, because why not?  \n- Kristen’s Corner, a quarterly reflection on educational wisdom.  \n- LearnStorm, their joyous classroom tool to keep student progress on the up and up.  \n\nAll in all, it\'s a platform for educators, learners, and parents to geek out on AI education and feel good about 

In [8]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [9]:
display_summary("https://blog.khanacademy.org/")

NameError: name 'Markdown' is not defined